# 🏠 Predicting House Prices with Linear Regression
**Oasis Infobyte SIP · Data Analytics · Level 2 · Task 1**

**Objective:** Build and evaluate a Linear Regression model that predicts house prices from features such as area, rooms, amenities and location-related flags — covering the full pipeline from data cleaning to model interpretation.

**Dataset:** *Housing Prices Dataset* (`Housing.csv`, 545 rows) from Kaggle. Place the file at `data/Housing.csv`.

**Pipeline:**
1. Load data & EDA
2. Feature selection discussion
3. Missing values & encoding
4. Correlation heatmap
5. Train/test split (80/20)
6. Linear Regression
7. Evaluation (MSE, RMSE, R²)
8. Actual vs. predicted plot
9. Residual analysis
10. Coefficient analysis
11. Bonus: Ridge & Lasso comparison

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

## 2. Load Data

In [ ]:
DATA_PATH = next((p for p in [Path("data/Housing.csv"), Path("Housing.csv")] if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Housing.csv not found. Download it from Kaggle and place it in the data/ folder.")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Structure, data types, nulls and duplicates

In [ ]:
print("Shape:", df.shape)
print("\nData types & non-null counts:")
df.info()

In [ ]:
print("Null values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

### 3.2 Descriptive statistics

In [ ]:
display(df.describe().T)
cat_cols_all = df.select_dtypes(exclude="number").columns.tolist()
print("\nCategorical columns:", cat_cols_all)
display(df[cat_cols_all].describe().T)

### 3.3 Distribution of the target variable (price)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["price"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of House Price")
sns.boxplot(x=df["price"], ax=axes[1], color="lightcoral")
axes[1].set_title("Price Boxplot (outlier check)")
plt.tight_layout(); plt.show()

print(f"Mean price   : {df['price'].mean():,.0f}")
print(f"Median price : {df['price'].median():,.0f}")
print(f"Skewness     : {df['price'].skew():.2f}")

**Observation guide:** If the mean is noticeably higher than the median and skewness is positive, the price distribution is *right-skewed* — a small number of expensive houses pull the tail out. Linear Regression can still work, but this is a hint that errors may be larger for high-priced homes (check the residual plot later).

### 3.4 Numeric features

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns if c != "price"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], kde=False, ax=ax, color="teal", bins=20)
    ax.set_title(col)
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="area", y="price", ax=axes[0], alpha=0.6)
axes[0].set_title("Area vs Price")
sns.boxplot(data=df, x="bedrooms", y="price", ax=axes[1])
axes[1].set_title("Bedrooms vs Price")
plt.tight_layout(); plt.show()

### 3.5 Categorical features vs. price

In [ ]:
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
n = len(cat_cols)
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.boxplot(data=df, x=col, y="price", ax=ax)
    ax.set_title(f"Price by {col}")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Feature Selection Discussion

Which features are likely predictors of house price, and why?

| Feature | Expected effect | Reasoning |
|---|---|---|
| `area` | ↑ strongly positive | Larger plots/floor area directly increase value; usually the single strongest predictor. |
| `bathrooms`, `bedrooms`, `stories` | ↑ positive | More rooms and floors mean more usable space. Bathrooms are often a stronger signal than bedrooms. |
| `airconditioning`, `hotwaterheating`, `basement`, `guestroom` | ↑ positive | Amenities that raise comfort and buyer demand. |
| `parking` | ↑ positive | Parking spaces add convenience and value. |
| `prefarea`, `mainroad` | ↑ positive | **Location proxies** — a preferred area and main-road access usually carry a price premium. |
| `furnishingstatus` | mixed | Furnished homes may sell at a premium; unfurnished at a discount. |

**Notes on the dataset:**
- The dataset has no exact *location* or *age* column. `prefarea` and `mainroad` act as the location signal.
- Features are kept in the model unless they are redundant; with only ~12 predictors, Linear Regression handles all of them without feature elimination. Coefficient analysis (Section 11) shows which ones matter most.

## 5. Handle Missing Values & Encode Categorical Features

In [ ]:
df_clean = df.copy()

# 5.1 Missing values: median for numeric, mode for categorical (no-op if there are none)
for col in df_clean.columns:
    if df_clean[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        else:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
print("Remaining nulls:", df_clean.isnull().sum().sum())

# 5.2 Remove duplicates (if any)
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print("Duplicates removed:", before - len(df_clean))

# 5.3 Outliers: report using IQR, but RETAIN them (they are real, expensive houses)
Q1, Q3 = df_clean["price"].quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = df_clean[(df_clean["price"] > Q3 + 1.5 * IQR) | (df_clean["price"] < Q1 - 1.5 * IQR)]
print(f"Price outliers (IQR rule): {len(outliers)} — retained as genuine high-value properties")

In [ ]:
# 5.4 Encoding
cat_cols = df_clean.select_dtypes(exclude="number").columns.tolist()

# Binary yes/no columns -> 1/0
binary_cols = [c for c in cat_cols if set(df_clean[c].dropna().unique()) <= {"yes", "no"}]
for c in binary_cols:
    df_clean[c] = df_clean[c].map({"yes": 1, "no": 0})

# Remaining multi-class categoricals -> One-Hot Encoding (drop_first avoids the dummy-variable trap)
multi_cols = [c for c in cat_cols if c not in binary_cols]
df_enc = pd.get_dummies(df_clean, columns=multi_cols, drop_first=True, dtype=int)

print("Binary-encoded :", binary_cols)
print("One-hot encoded:", multi_cols)
print("Final shape    :", df_enc.shape)
df_enc.head()

## 6. Correlation Heatmap

In [ ]:
corr = df_enc.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix")
plt.tight_layout(); plt.show()

price_corr = corr["price"].drop("price").sort_values(ascending=False)
print("Correlation with price (highest to lowest):")
print(price_corr.round(3))
print(f"\n➡ Most correlated feature: '{price_corr.index[0]}' (r = {price_corr.iloc[0]:.2f})")

**Observation guide:** Features with the largest absolute correlation with `price` are the most promising predictors. Also check for high correlation *between* features (e.g. bedrooms vs. bathrooms) — that is multicollinearity, which can make individual coefficients unstable even if overall predictions are fine.

## 7. Train/Test Split (80/20)

In [ ]:
X = df_enc.drop(columns="price")
y = df_enc["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 8. Train the Linear Regression Model

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train = lr.predict(X_train)
y_pred_test = lr.predict(X_test)
print("Model trained.  Intercept:", round(lr.intercept_, 2))

## 9. Model Evaluation (MSE, RMSE, R²)

In [ ]:
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MSE": mse, "RMSE": np.sqrt(mse), "MAE": mean_absolute_error(y_true, y_pred), "R²": r2_score(y_true, y_pred)}

results = pd.DataFrame({
    "Train": evaluate(y_train, y_pred_train),
    "Test": evaluate(y_test, y_pred_test),
}).T
display(results.style.format({"MSE": "{:,.0f}", "RMSE": "{:,.0f}", "MAE": "{:,.0f}", "R²": "{:.4f}"}))

test_r2 = results.loc["Test", "R²"]
print(f"\nThe model explains {test_r2*100:.1f}% of the variance in house prices on unseen data.")
print(f"Typical prediction error (RMSE): {results.loc['Test','RMSE']:,.0f} "
      f"({results.loc['Test','RMSE']/y_test.mean()*100:.1f}% of the mean test price)")

**How to read this:**
- **MSE** – average squared error; penalises large mistakes heavily (units are price²).
- **RMSE** – square root of MSE, in the same units as price; the "typical" error size.
- **R²** – share of price variance explained (1.0 = perfect, 0 = no better than predicting the mean).
- Compare **Train vs. Test**: similar scores mean the model generalises; a much higher train score suggests overfitting.

## 10. Actual vs. Predicted Prices

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_test, alpha=0.65, edgecolor="k", linewidth=0.3)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, "r--", label="Perfect prediction")
ax.set_xlabel("Actual Price"); ax.set_ylabel("Predicted Price")
ax.set_title("Actual vs Predicted (Test Set)")
ax.legend(); plt.tight_layout(); plt.show()

**Observation guide:** The closer the points sit to the red diagonal, the better the predictions. Look for systematic drift at the high end — that would mean the model under-predicts expensive houses.

## 11. Residual Analysis

In [ ]:
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(y_pred_test, residuals, alpha=0.65, edgecolor="k", linewidth=0.3)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted Price"); axes[0].set_ylabel("Residual (Actual − Predicted)")
axes[0].set_title("Residuals vs Predicted")

sns.histplot(residuals, kde=True, ax=axes[1], color="purple")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Distribution of Residuals")
plt.tight_layout(); plt.show()

print(f"Mean residual: {residuals.mean():,.0f}  (should be close to 0)")
print(f"Residual skewness: {residuals.skew():.2f}")

**What to check:**
- **Random scatter around 0** with no pattern → the linear model is appropriate.
- A **funnel shape** (spread growing with predicted price) → heteroscedasticity; consider modelling `log(price)`.
- A **curve** → non-linear relationships the model is missing.
- A roughly **bell-shaped, centred histogram** → errors are approximately normal.

## 12. Coefficient Analysis — Which Features Drive Price?

In [ ]:
coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": lr.coef_}).sort_values("Coefficient", ascending=False)
display(coef_df.style.format({"Coefficient": "{:,.2f}"}))

In [ ]:
# Raw coefficients are in different units (area in sq. ft vs. 0/1 flags), so they are not directly comparable.
# Standardise X so each coefficient = price change per +1 standard deviation of the feature.
scaler = StandardScaler().fit(X_train)
lr_std = LinearRegression().fit(scaler.transform(X_train), y_train)
std_coef = pd.Series(lr_std.coef_, index=X.columns).sort_values()

plt.figure(figsize=(8, 6))
colors = ["seagreen" if v > 0 else "indianred" for v in std_coef]
std_coef.plot(kind="barh", color=colors)
plt.title("Standardised Coefficients (impact of +1 SD on price)")
plt.xlabel("Change in price"); plt.axvline(0, color="k", linewidth=0.8)
plt.tight_layout(); plt.show()

print("Top 3 positive drivers :", std_coef.sort_values(ascending=False).head(3).index.tolist())
print("Top 3 negative drivers :", std_coef.sort_values().head(3).index.tolist())

**Interpretation:** A positive coefficient means that, holding other features constant, increasing the feature raises the predicted price; a negative one lowers it. Use the **standardised** chart to rank importance and the **raw** table to quote real-world effects (e.g. "one extra bathroom adds ≈ X to the price").

⚠️ Linear Regression coefficients show **association, not causation**, and correlated features (multicollinearity) can share or distort credit.

## 13. 🎁 Bonus: Linear vs. Ridge vs. Lasso

In [ ]:
models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (CV)": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 40))),
    "Lasso (CV)": make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=100000)),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    m = evaluate(y_test, model.predict(X_test))
    rows.append({"Model": name, **m})
comparison = pd.DataFrame(rows).set_index("Model")
display(comparison.style.format({"MSE": "{:,.0f}", "RMSE": "{:,.0f}", "MAE": "{:,.0f}", "R²": "{:.4f}"}))

best = comparison["R²"].idxmax()
print(f"Best test R²: {best}")
lasso = models["Lasso (CV)"].named_steps["lassocv"]
print("Features zeroed out by Lasso:", list(X.columns[lasso.coef_ == 0]) or "none")

**Takeaway:** Ridge shrinks coefficients to reduce the effect of multicollinearity; Lasso can shrink some to exactly zero (built-in feature selection). If all three score almost the same, plain Linear Regression is the simplest, most interpretable choice.

## 14. Conclusion

In [ ]:
top_feat = std_coef.abs().sort_values(ascending=False).head(3).index.tolist()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Dataset            : {df.shape[0]} houses, {X.shape[1]} features after encoding")
print(f"Test R²            : {results.loc['Test','R²']:.3f}")
print(f"Test RMSE          : {results.loc['Test','RMSE']:,.0f}")
print(f"Top price drivers  : {', '.join(top_feat)}")
print(f"Best bonus model   : {best} (R² = {comparison['R²'].max():.3f})")

### Key takeaways
1. **Model performance:** Linear Regression gives a solid, interpretable baseline. See the R² and RMSE above for how much of the price variation it captures.
2. **Main drivers:** Size (`area`), bathrooms and amenities such as air-conditioning and preferred location typically dominate — confirm with your standardised coefficient chart.
3. **Limitations:** The dataset is small (~545 rows), has no exact location or house age, and prices are right-skewed. Residual patterns (if any) show where a linear model falls short.
4. **Next steps:** Try `log(price)` as the target, add interaction terms (e.g. `area × prefarea`), or test non-linear models like Random Forest or Gradient Boosting.